In [ ]:
import numpy as np


def rms_norm(x: np.ndarray, weight: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    RMSNorm: x * w / sqrt(mean(x^2) + eps)
    x: (batch_size, seq_len, dim)
    weight: (dim,)
    """
    variance = np.mean(x**2, axis=-1, keepdims=True)
    return (x / np.sqrt(variance + eps)) * weight


def precompute_freqs_cis(head_dim: int, max_seq_len: int = 2048, base: float = 10000.0) -> np.ndarray:
    """
    预计算 RoPE 复数旋转因子表: exp(i * m * theta)
    返回 shape: (max_seq_len, head_dim // 2), dtype: complex64
    """
    dim_indices = np.arange(0, head_dim, 2, dtype=np.float32)
    inv_freq = 1.0 / (base ** (dim_indices / head_dim))
    positions = np.arange(max_seq_len, dtype=np.float32)
    angles = np.outer(positions, inv_freq)
    return np.exp(1j * angles).astype(np.complex64)


def apply_rotary_emb(x: np.ndarray, freqs_cis: np.ndarray) -> np.ndarray:
    """
    应用 RoPE 旋转位置编码 (使用 complex64 零拷贝视图)
    x: (batch_size, num_heads, seq_len, head_dim)
    freqs_cis: (seq_len, head_dim // 2)
    """
    # 转换为复数视图: (B, H, S, D // 2)
    x_complex = x.astype(np.float32).view(np.complex64)
    # 调整 freqs_cis 形状以适配多头广播: (1, 1, seq_len, head_dim // 2)
    freqs_cis = freqs_cis[None, None, :, :]
    # 执行复数乘法旋转并恢复 float32 视图
    x_rotated = (x_complex * freqs_cis).view(np.float32)
    return x_rotated


def silu(x: np.ndarray) -> np.ndarray:
    return x / (1.0 + np.exp(-x))


class KVCache:
    def __init__(self):
        self.k: np.ndarray = None  # (batch_size, num_heads, total_seq_len, head_dim)
        self.v: np.ndarray = None  # (batch_size, num_heads, total_seq_len, head_dim)

    def update(self, k_val: np.ndarray, v_val: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        k_val, v_val: (batch_size, num_heads, cur_seq_len, head_dim)
        沿着序列长度轴 (axis=2) 动态拼接历史缓存
        """
        if self.k is None:
            self.k = k_val
            self.v = v_val
        else:
            self.k = np.concatenate([self.k, k_val], axis=2)
            self.v = np.concatenate([self.v, v_val], axis=2)
        return self.k, self.v

    @property
    def current_len(self) -> int:
        return 0 if self.k is None else self.k.shape[2]


class TransformerDecoderLayer:
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # 预计算位置编码表
        self.freqs_cis_table = precompute_freqs_cis(self.head_dim, max_seq_len=2048)

        # 1. 权重初始化 (使用小型正态分布)
        scale = 1.0 / np.sqrt(d_model)
        self.wq = np.random.randn(d_model, d_model) * scale
        self.wk = np.random.randn(d_model, d_model) * scale
        self.wv = np.random.randn(d_model, d_model) * scale
        self.wo = np.random.randn(d_model, d_model) * scale

        # SwiGLU MLP 权重
        self.w_gate = np.random.randn(d_model, d_ff) * scale
        self.w_up = np.random.randn(d_model, d_ff) * scale
        self.w_down = np.random.randn(d_ff, d_model) * (1.0 / np.sqrt(d_ff))

        # 归一化参数
        self.attn_norm_w = np.ones(d_model, dtype=np.float32)
        self.ffn_norm_w = np.ones(d_model, dtype=np.float32)

    def forward(self, x: np.ndarray, kv_cache: KVCache, start_pos: int) -> np.ndarray:
        """
        x: (batch_size, seq_len, d_model)
        start_pos: 当前输入 token 在整个序列中的起始绝对索引
        """
        B, S, D = x.shape
        H, HD = self.num_heads, self.head_dim

        # ----------------- 1. Self-Attention 阶段 -----------------
        residual = x
        h = rms_norm(x, self.attn_norm_w)

        # 投影到 Q, K, V 并切分为多头 (B, S, D) -> (B, H, S, HD)
        q = (h @ self.wq).reshape(B, S, H, HD).transpose(0, 2, 1, 3)
        k = (h @ self.wk).reshape(B, S, H, HD).transpose(0, 2, 1, 3)
        v = (h @ self.wv).reshape(B, S, H, HD).transpose(0, 2, 1, 3)

        # 应用 RoPE (根据当前 token 的全局绝对位置切片旋转因子)
        cur_freqs_cis = self.freqs_cis_table[start_pos : start_pos + S]
        q = apply_rotary_emb(q, cur_freqs_cis)
        k = apply_rotary_emb(k, cur_freqs_cis)

        # 写入并更新 KV Cache: 返回全量 (历史 + 当前) 的 K, V
        # shape: (B, H, Total_S, HD)
        keys, values = kv_cache.update(k, v)
        total_s = keys.shape[2]

        # 计算 Attention Scores: Q @ K^T / sqrt(d_k)
        # q: (B, H, S, HD), keys.T: (B, H, HD, Total_S) -> (B, H, S, Total_S)
        scores = (q @ keys.swapaxes(-1, -2)) / np.sqrt(HD)

        # 掩码机制 (Prefill 阶段需要下三角因果 Mask，Decode 阶段 1 个 Token 看全局无需 Mask)
        if S > 1:
            # 构造因果掩码: 当前行只能看到该行之前的列
            mask = np.full((S, total_s), float("-inf"), dtype=np.float32)
            mask = np.triu(mask, k=total_s - S + 1)
            scores = scores + mask[None, None, :, :]

        # 数值稳定 Softmax
        scores_max = np.max(scores, axis=-1, keepdims=True)
        attn_weights = np.exp(scores - scores_max)
        attn_weights /= np.sum(attn_weights, axis=-1, keepdims=True)

        # 加权求和 (B, H, S, Total_S) @ (B, H, Total_S, HD) -> (B, H, S, HD)
        attn_out = attn_weights @ values

        # 还原维度 (B, H, S, HD) -> (B, S, D) 并做输出投影
        attn_out = attn_out.transpose(0, 2, 1, 3).reshape(B, S, D)
        h = residual + (attn_out @ self.wo)

        # ----------------- 2. Feed-Forward (SwiGLU) 阶段 -----------------
        residual = h
        h_norm = rms_norm(h, self.ffn_norm_w)

        # SwiGLU: (silu(x @ W_gate) * (x @ W_up)) @ W_down
        gate = silu(h_norm @ self.w_gate)
        up = h_norm @ self.w_up
        ffn_out = (gate * up) @ self.w_down

        out = residual + ffn_out
        return out


# 设定超参数
batch_size = 1
d_model = 64
num_heads = 4
d_ff = 128
prefill_len = 5  # Prompt 包含 5 个 Token
gen_steps = 3  # 自回归生成 3 个 Token

# 初始化层与缓存
np.random.seed(42)
layer = TransformerDecoderLayer(d_model=d_model, num_heads=num_heads, d_ff=d_ff)
kv_cache = KVCache()

# =======================
# 阶段 1: Prefill (预填充阶段)
# =======================
print("=== 1. 进入 Prefill 阶段 ===")
prompt_tokens = np.random.randn(batch_size, prefill_len, d_model).astype(np.float32)

start_pos = 0
out_prefill = layer.forward(prompt_tokens, kv_cache, start_pos=start_pos)

print(f"Prompt 形状: {prompt_tokens.shape}")
print(f"Prefill 输出形状: {out_prefill.shape}")
print(f"Prefill 完成后 KV Cache 累积长度: {kv_cache.current_len}")

# =======================
# 阶段 2: Decode (逐字生成阶段)
# =======================
print("\n=== 2. 进入 Decode 自回归阶段 ===")
current_token = np.random.randn(batch_size, 1, d_model).astype(np.float32)

for step in range(gen_steps):
    start_pos = kv_cache.current_len  # 当前 Token 的位置索引

    # 每次只输入 1 个 Token (seq_len = 1)
    out_step = layer.forward(current_token, kv_cache, start_pos=start_pos)

    print(
        f"Step {step + 1}: 输入 Token 位置 = {start_pos}, "
        f"输出形状 = {out_step.shape}, "
        f"更新后 KV Cache 总长度 = {kv_cache.current_len}"
    )

    # 模拟下一轮输入（真实模型中会取 argmax / 采样并查 Embedding 表）
    current_token = out_step